## Этап 1 - Подключение библиотек и базовые настройки

В этом разделе подключаются библиотеки для загрузки и обработки табличных данных, работы с геометрией объектов, построения пайплайна предобработки и обучения модели градиентного бустинга. Также импортируются инструменты для кросс-валидации, расчета метрик качества и перебора гиперпараметров.

Цель текущего эксперимента — построить устойчивую модель предсказания высоты здания на основе признаков, полученных после пространственного объединения данных. Отдельное внимание уделяется корректной валидации: оценка качества должна проводиться с учетом пространственной структуры выборки, чтобы избежать утечки информации между близко расположенными объектами.

In [7]:
import numpy as np
import pandas as pd

from shapely import wkb

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

from itertools import product

In [8]:
DATA_PATH = "merged_buildings_by_geometry.parquet"
RANDOM_STATE = 42
N_SPLITS = 5
GRID_SIZE = 1000

main_cols = [
    "component_id",
    "match_type",
    "match_confidence",
    "geometry_source",

    "target_height",
    "target_height_is_observed",
    "target_height_source",
    "target_height_source_detail",
    "target_height_reliability",

    "n_a",
    "n_b",
    "uids_a",
    "uids_b",

    "n_edges_ab",
    "max_iou",
    "mean_iou",
    "max_overlap_a",
    "max_overlap_b",
    "min_dist_m",

    "sum_area_a",
    "sum_area_b",
    "union_area_a",
    "union_area_b",
    "union_area_all",

    "n_b_with_height",
    "median_height_b",
    "median_stairs_b",
    "median_avg_floor_height_b",
    "mode_purpose_b",

    "n_neighbors_50m",
    "n_neighbors_obs_height_50m",
    "neighbor_height_mean_50m",
    "neighbor_height_median_50m",
    "neighbor_height_min_50m",
    "neighbor_height_max_50m",
    "neighbor_height_std_50m",
    "neighbor_height_q25_50m",
    "neighbor_height_q75_50m",

    "n_neighbors_100m",
    "n_neighbors_obs_height_100m",
    "neighbor_height_mean_100m",
    "neighbor_height_median_100m",
    "neighbor_height_min_100m",
    "neighbor_height_max_100m",
    "neighbor_height_std_100m",
    "neighbor_height_q25_100m",
    "neighbor_height_q75_100m",

    "rep_geometry",
]

## Этап 2 - Подготовка к обучению

**Подготовка геометрии и пространственных групп**

В этой части описываются вспомогательные функции для работы с геометрией объектов. Геометрия хранится в бинарном формате WKB, поэтому сначала требуется преобразовать ее в объекты `shapely`, пригодные для дальнейших вычислений.

После восстановления геометрии для каждого объекта вычисляется центроид. Координаты центроидов используются для разбиения пространства на укрупненную регулярную сетку. Каждому объекту назначается пространственная группа по номеру ячейки сетки. Именно эти группы затем используются в `GroupKFold`, чтобы близкие по расположению здания не попадали одновременно в обучающую и тестовую части одного фолда.

---

**Обработка пропусков в соседских признаках**

Признаки, связанные с высотами соседних объектов, могут содержать пропуски. Это происходит в ситуациях, когда в заданном радиусе не найдено ни одного соседа с наблюдаемой высотой либо статистика не может быть вычислена корректно.

Чтобы не терять информацию о самой природе пропуска, для каждого такого признака создается отдельный бинарный индикатор наличия значения. После этого пропуски в числовых соседских статистиках временно заменяются на нули. В результате модель получает сразу два сигнала: само числовое значение и информацию о том, было ли оно реально наблюдено.

---

**Метрики качества и схема валидации**

Для оценки качества модели используются стандартные метрики регрессии: `MSE`, `RMSE`, `MAE` и `R²`. Они позволяют смотреть на ошибку с разных сторон: среднеквадратично, в исходном масштабе целевой переменной, по абсолютному отклонению и по доле объясненной дисперсии.

Кросс-валидация проводится не случайным образом, а с использованием `GroupKFold`. В качестве групп выступают пространственные ячейки, полученные на предыдущем шаге. Это делает оценку более честной: модель проверяется не на почти тех же самых соседних объектах, а на пространственно отделенных подвыборках.

**Формирование альтернативных наборов признаков**

На данном этапе собирается несколько вариантов признакового описания объекта. Это нужно для того, чтобы сравнить, как меняется качество модели при постепенном расширении информационной базы.

Базовый набор включает только самые общие характеристики: этажность, среднюю высоту этажа, площадь объединенной геометрии и назначение объекта. Далее к нему добавляются агрегаты по числу источников и ребер сопоставления, затем — средние соседские признаки, а после этого — полный набор статистик по соседям в радиусах 50 и 100 метров.

Отдельно формируется максимально полный набор признаков, включающий как структурные характеристики объединения объектов, так и пространственные статистики окружения. Такое многоуровневое сравнение позволяет понять, какие группы признаков действительно улучшают предсказание высоты.





In [9]:
def load_wkb_geometry(value):
    if value is None or pd.isna(value):
        return None
    
    if isinstance(value, memoryview):
        value = value.tobytes()
    elif isinstance(value, bytearray):
        value = bytes(value)
    
    return wkb.loads(value)


def build_spatial_groups(df, geom_col="rep_geometry", grid_size=1000):
    geoms = df[geom_col].apply(load_wkb_geometry)
    centroids = geoms.apply(lambda g: g.centroid if g is not None else None)

    x = centroids.apply(lambda c: c.x if c is not None else np.nan)
    y = centroids.apply(lambda c: c.y if c is not None else np.nan)

    grid_x = np.floor(x / grid_size).astype("Int64")
    grid_y = np.floor(y / grid_size).astype("Int64")

    groups = grid_x.astype(str) + "_" + grid_y.astype(str)
    return groups, x, y


def add_neighbor_missing_indicators(df):
    df = df.copy()

    neighbor_stat_cols = [
        c for c in df.columns
        if c.startswith("neighbor_height_")
    ]

    for col in neighbor_stat_cols:
        ind_col = f"has_{col}"
        df[ind_col] = df[col].notna().astype(int)
        df[col] = df[col].fillna(0)

    return df


def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "rmse": rmse,
        "mae": mae,
        "mse": mse,
        "r2": r2,
    }


def cross_validate_with_groups(
    model,
    X,
    y,
    groups,
    n_splits=5,
    return_oof=True,
):
    cv = GroupKFold(n_splits=n_splits)

    fold_rows = []
    oof_pred = np.full(len(X), np.nan, dtype=float)

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model_fold = clone(model)
        model_fold.fit(X_train, y_train)
        pred = model_fold.predict(X_test)

        if return_oof:
            oof_pred[test_idx] = pred

        metrics = compute_metrics(y_test, pred)
        metrics["fold"] = fold
        fold_rows.append(metrics)

    folds_df = pd.DataFrame(fold_rows)
    mean_metrics = folds_df[["rmse", "mae", "mse", "r2"]].mean().to_dict()

    return folds_df, mean_metrics, oof_pred


def make_feature_sets():
    baseline = [
        "median_stairs_b",
        "median_avg_floor_height_b",
        "union_area_all",
        "mode_purpose_b",
    ]

    plus_n_ab = baseline + [
        "n_a",
        "n_b",
        "n_b_with_height",
        "n_edges_ab",
    ]

    neighbors_mean = plus_n_ab + [
        "n_neighbors_50m",
        "n_neighbors_obs_height_50m",
        "neighbor_height_mean_50m",
        "has_neighbor_height_mean_50m",
        "n_neighbors_100m",
        "n_neighbors_obs_height_100m",
        "neighbor_height_mean_100m",
        "has_neighbor_height_mean_100m",
    ]

    neighbors_full_50m = plus_n_ab + [
        "n_neighbors_50m",
        "n_neighbors_obs_height_50m",
        "neighbor_height_mean_50m",
        "neighbor_height_median_50m",
        "neighbor_height_min_50m",
        "neighbor_height_max_50m",
        "neighbor_height_std_50m",
        "neighbor_height_q25_50m",
        "neighbor_height_q75_50m",
        "has_neighbor_height_mean_50m",
        "has_neighbor_height_median_50m",
        "has_neighbor_height_min_50m",
        "has_neighbor_height_max_50m",
        "has_neighbor_height_std_50m",
        "has_neighbor_height_q25_50m",
        "has_neighbor_height_q75_50m",
    ]

    neighbors_full_100m = plus_n_ab + [
        "n_neighbors_100m",
        "n_neighbors_obs_height_100m",
        "neighbor_height_mean_100m",
        "neighbor_height_median_100m",
        "neighbor_height_min_100m",
        "neighbor_height_max_100m",
        "neighbor_height_std_100m",
        "neighbor_height_q25_100m",
        "neighbor_height_q75_100m",
        "has_neighbor_height_mean_100m",
        "has_neighbor_height_median_100m",
        "has_neighbor_height_min_100m",
        "has_neighbor_height_max_100m",
        "has_neighbor_height_std_100m",
        "has_neighbor_height_q25_100m",
        "has_neighbor_height_q75_100m",
    ]

    full = [
        "match_type",
        "match_confidence",
        "geometry_source",
        "target_height_reliability",
        "n_a",
        "n_b",
        "n_edges_ab",
        "sum_area_a",
        "sum_area_b",
        "union_area_a",
        "union_area_b",
        "union_area_all",
        "n_b_with_height",
        "median_height_b",
        "median_stairs_b",
        "median_avg_floor_height_b",
        "mode_purpose_b",

        "n_neighbors_50m",
        "n_neighbors_obs_height_50m",
        "neighbor_height_mean_50m",
        "neighbor_height_median_50m",
        "neighbor_height_min_50m",
        "neighbor_height_max_50m",
        "neighbor_height_std_50m",
        "neighbor_height_q25_50m",
        "neighbor_height_q75_50m",

        "n_neighbors_100m",
        "n_neighbors_obs_height_100m",
        "neighbor_height_mean_100m",
        "neighbor_height_median_100m",
        "neighbor_height_min_100m",
        "neighbor_height_max_100m",
        "neighbor_height_std_100m",
        "neighbor_height_q25_100m",
        "neighbor_height_q75_100m",

        "has_neighbor_height_mean_50m",
        "has_neighbor_height_median_50m",
        "has_neighbor_height_min_50m",
        "has_neighbor_height_max_50m",
        "has_neighbor_height_std_50m",
        "has_neighbor_height_q25_50m",
        "has_neighbor_height_q75_50m",

        "has_neighbor_height_mean_100m",
        "has_neighbor_height_median_100m",
        "has_neighbor_height_min_100m",
        "has_neighbor_height_max_100m",
        "has_neighbor_height_std_100m",
        "has_neighbor_height_q25_100m",
        "has_neighbor_height_q75_100m",
    ]

    return {
        "baseline": baseline,
        "plus_n_ab": plus_n_ab,
        "neighbors_mean": neighbors_mean,
        "neighbors_full_50m": neighbors_full_50m,
        "neighbors_full_100m": neighbors_full_100m,
        "full": full,
    }

## Этап 3 - Построение пайплайна модели

Для обучения используется `HistGradientBoostingRegressor` — градиентный бустинг по гистограммам, хорошо работающий на табличных данных и устойчивый по времени обучения на средних и больших выборках.

Перед моделью строится единый пайплайн предобработки. Числовые признаки проходят медианную импутацию, а категориальные — заполнение наиболее частым значением и порядковое кодирование через `OrdinalEncoder`. Категориальные переменные не переводятся в one-hot-представление, чтобы не увеличивать размерность пространства признаков без необходимости.

Такой пайплайн обеспечивает единообразную обработку данных внутри каждого фолда и предотвращает утечки, так как все преобразования обучаются только на тренировочной части.

---

**Подготовка целевой переменной, групп и запуск экспериментов**

Перед серией экспериментов отдельно выделяются целевая переменная и столбец пространственных групп. После этого для каждого набора признаков последовательно запускается обучение моделей на всех конфигурациях гиперпараметров.

Для каждой комбинации выполняется групповая кросс-валидация, рассчитываются метрики качества и сохраняются out-of-fold предсказания. Такой формат удобен тем, что по итогам можно не только сравнить средние значения метрик, но и при необходимости провести дальнейший анализ ошибок уже на уровне отдельных объектов.

---

**Сбор результатов в таблицу**

Результаты всех прогонов собираются в единую таблицу, после чего сортируются по ключевым метрикам качества. Это позволяет быстро увидеть лучшие конфигурации модели и оценить, какие наборы признаков дают наибольший вклад в точность.

In [10]:
df = pd.read_parquet(DATA_PATH)
df = df[main_cols].copy()

df = df[df["target_height_is_observed"] == 1].copy()

drop_match_metric_cols = [
    "max_iou",
    "mean_iou",
    "max_overlap_a",
    "max_overlap_b",
    "min_dist_m",
]
df = df.drop(columns=drop_match_metric_cols)

df = add_neighbor_missing_indicators(df)

df["spatial_group"], df["centroid_x"], df["centroid_y"] = build_spatial_groups(
    df,
    geom_col="rep_geometry",
    grid_size=GRID_SIZE,
)

df = df[df["spatial_group"].notna()].copy()

print(df.shape)
df.head()

(101563, 60)


,component_id,match_type,match_confidence,geometry_source,target_height,target_height_is_observed,target_height_source,target_height_source_detail,target_height_reliability,n_a,...,has_neighbor_height_mean_100m,has_neighbor_height_median_100m,has_neighbor_height_min_100m,has_neighbor_height_max_100m,has_neighbor_height_std_100m,has_neighbor_height_q25_100m,has_neighbor_height_q75_100m,spatial_group,centroid_x,centroid_y
0,1,n:1,high,A,4.50,1,B_observed,B_from_nto1_match,medium,8,...,1,1,1,1,1,1,1,673_6635,673859.539664,6.635505e+06
1,2,n:1,high,A,4.50,1,B_observed,B_from_nto1_match,medium,5,...,1,1,1,1,1,1,1,673_6635,673885.218595,6.635511e+06
2,3,n:n,high,A,60.00,1,B_observed,B_from_nton_match,medium,12,...,1,1,1,1,1,1,1,677_6640,677022.530648,6.640407e+06
5,6,n:n,high,B,48.00,1,B_observed,B_from_nton_match,medium,18,...,1,1,1,1,1,1,1,678_6638,678532.626405,6.638733e+06
6,7,n:n,high,B,68.25,1,B_observed,B_from_nton_match,medium,3,...,1,1,1,1,1,1,1,678_6640,678893.429480,6.640099e+06


In [ ]:
def build_hgb_pipeline(feature_cols, hgb_params):
    X_sample = df[feature_cols]

    categorical_cols = X_sample.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                ]),
                numeric_cols,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("ord", OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1
                    )),
                ]),
                categorical_cols,
            ),
        ],
        remainder="drop",
    )

    model = HistGradientBoostingRegressor(
        random_state=RANDOM_STATE,
        **hgb_params,
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    return pipe

In [12]:
hgb_param_grid = {
    "learning_rate": [0.03, 0.05],
    "max_iter": [300],
    "max_depth": [None, 8],
    "min_samples_leaf": [20, 50],
    "l2_regularization": [0.0, 1.0],
}

hgb_param_list = [
    dict(zip(hgb_param_grid.keys(), values))
    for values in product(*hgb_param_grid.values())
]

print(f"Всего комбинаций HGB: {len(hgb_param_list)}")
hgb_param_list[:3]

Всего комбинаций HGB: 16


[{'learning_rate': 0.03,
  'max_iter': 300,
  'max_depth': None,
  'min_samples_leaf': 20,
  'l2_regularization': 0.0},
 {'learning_rate': 0.03,
  'max_iter': 300,
  'max_depth': None,
  'min_samples_leaf': 20,
  'l2_regularization': 1.0},
 {'learning_rate': 0.03,
  'max_iter': 300,
  'max_depth': None,
  'min_samples_leaf': 50,
  'l2_regularization': 0.0}]

In [13]:
feature_sets = make_feature_sets()

target_col = "target_height"
group_col = "spatial_group"

print("Наборы признаков:", list(feature_sets.keys()))
print("target_col =", target_col)
print("group_col =", group_col)

Наборы признаков: ['baseline', 'plus_n_ab', 'neighbors_mean', 'neighbors_full_50m', 'neighbors_full_100m', 'full']
target_col = target_height
group_col = spatial_group


In [14]:
hgb_results = []
hgb_oof_store = {}

y = df[target_col].reset_index(drop=True)
groups = df[group_col].reset_index(drop=True)

for set_name, feature_cols in feature_sets.items():
    print(f"\n===== Feature set: {set_name} =====")
    
    X = df[feature_cols].reset_index(drop=True)

    for params in hgb_param_list:
        print(f"HGB params: {params}")

        pipe = build_hgb_pipeline(feature_cols, params)

        folds_df, mean_metrics, oof_pred = cross_validate_with_groups(
            model=pipe,
            X=X,
            y=y,
            groups=groups,
            n_splits=N_SPLITS,
            return_oof=True,
        )

        row = {
            "model_type": "hist_gradient_boosting",
            "feature_set": set_name,
            **params,
            **mean_metrics,
        }
        hgb_results.append(row)

        run_key = (
            "hist_gradient_boosting",
            set_name,
            tuple(sorted(params.items()))
        )
        hgb_oof_store[run_key] = oof_pred

hgb_results_df = pd.DataFrame(hgb_results).sort_values(["rmse", "mae"]).reset_index(drop=True)
hgb_results_df.head(20)


===== Feature set: baseline =====
HGB params: {'learning_rate': 0.03, 'max_iter': 300, 'max_depth': None, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
HGB params: {'learning_rate': 0.03, 'max_iter': 300, 'max_depth': None, 'min_samples_leaf': 20, 'l2_regularization': 1.0}
HGB params: {'learning_rate': 0.03, 'max_iter': 300, 'max_depth': None, 'min_samples_leaf': 50, 'l2_regularization': 0.0}
HGB params: {'learning_rate': 0.03, 'max_iter': 300, 'max_depth': None, 'min_samples_leaf': 50, 'l2_regularization': 1.0}
HGB params: {'learning_rate': 0.03, 'max_iter': 300, 'max_depth': 8, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
HGB params: {'learning_rate': 0.03, 'max_iter': 300, 'max_depth': 8, 'min_samples_leaf': 20, 'l2_regularization': 1.0}
HGB params: {'learning_rate': 0.03, 'max_iter': 300, 'max_depth': 8, 'min_samples_leaf': 50, 'l2_regularization': 0.0}
HGB params: {'learning_rate': 0.03, 'max_iter': 300, 'max_depth': 8, 'min_samples_leaf': 50, 'l2_regularization': 1.0}
H

,model_type,feature_set,learning_rate,max_iter,max_depth,min_samples_leaf,l2_regularization,rmse,mae,mse,r2
0,hist_gradient_boosting,full,0.03,300,NaN,50,1.0,0.719828,0.034766,1.418817,0.989486
1,hist_gradient_boosting,full,0.03,300,8.0,50,1.0,0.720768,0.035858,1.419463,0.989480
2,hist_gradient_boosting,full,0.05,300,8.0,50,1.0,0.720925,0.033033,1.418528,0.989487
3,hist_gradient_boosting,full,0.05,300,NaN,50,1.0,0.721453,0.032270,1.418687,0.989484
4,hist_gradient_boosting,full,0.03,300,8.0,50,0.0,0.723700,0.035677,1.419681,0.989476
5,hist_gradient_boosting,full,0.05,300,NaN,50,0.0,0.723817,0.033232,1.419931,0.989473
6,hist_gradient_boosting,full,0.03,300,NaN,50,0.0,0.724035,0.035343,1.419751,0.989475
7,hist_gradient_boosting,full,0.05,300,8.0,50,0.0,0.724592,0.032348,1.419859,0.989473
8,hist_gradient_boosting,full,0.05,300,8.0,20,1.0,0.732426,0.041153,1.414967,0.989473
9,hist_gradient_boosting,full,0.03,300,8.0,20,1.0,0.735770,0.043681,1.416122,0.989460


## Объяснение результатов

По итогам перебора видно, что наилучшее качество достигается у модели `HistGradientBoostingRegressor` на полном наборе признаков. Это означает, что для задачи предсказания высоты важны не только базовые геометрические характеристики объекта, но и более широкий контекст: качество сопоставления, агрегаты по объединенной геометрии и статистики по окружению.

Небольшой разброс метрик между лучшими конфигурациями показывает, что модель работает достаточно стабильно, а качество не определяется случайной удачной настройкой одного параметра. При этом ухудшение результатов на урезанных наборах признаков подтверждает, что соседская и структурная информация действительно добавляет полезный сигнал.
